[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-01-langchain-fundamentals.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · LangChain Fundamentals — Architecture, Setup, and First Chat Call
**certified-journeys / llm-engineering-certified** · Day 1 · Foundations

> **Goal for today:** By the end of this notebook you can install LangChain, instantiate a `ChatOpenAI` model, send your first chat completion call, inspect the full response object, and understand how `HumanMessage`, `AIMessage`, and `SystemMessage` compose a conversation.


In [ ]:
%pip install -q langchain langchain-openai langchain-community python-dotenv

## Step 1 · What is LangChain and why does it exist?

LangChain is an open-source framework that sits between your code and LLM providers (OpenAI, Anthropic, Cohere, local models, …). It provides:

| Layer | What it does | Key classes |
|---|---|---|
| **Model I/O** | Unified interface to any chat/completion model | `ChatOpenAI`, `ChatAnthropic` |
| **Messages** | Typed message objects that structure conversations | `HumanMessage`, `AIMessage`, `SystemMessage` |
| **Chains / LCEL** | Compose model calls, parsers, and tools into pipelines | `|` operator, `RunnableSequence` |
| **Memory** | Persist conversation history across turns | `ConversationBufferMemory` |
| **Agents** | Let models decide which tools to call | `AgentExecutor` |

**Official docs:** https://python.langchain.com/docs/introduction/

Today we focus on the **Model I/O** layer — the foundation everything else builds on.


In [ ]:
# Load environment variables from a .env file that contains OPENAI_API_KEY.
# We fall back to a mock key so the notebook can be explored without a real key.
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env in the current directory (or parent dirs)

# Safety net: if no key was found in .env, set a placeholder so imports succeed.
# Replace 'sk-...' with your real key, or add it to a .env file.
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "sk-placeholder-replace-with-real-key"
    print("[INFO] Using placeholder API key — set OPENAI_API_KEY in your .env file.")
else:
    print("[OK] OPENAI_API_KEY loaded from environment.")

### What just happened?
- `load_dotenv()` reads a `.env` file and injects its contents into `os.environ`.
- **Never hardcode API keys in notebooks** — they end up in git history and Colab sharing links.
- The placeholder key lets you run subsequent cells without crashing on import.
- In production, use a secrets manager (AWS Secrets Manager, GCP Secret Manager) instead of `.env`.


## Step 2 · Instantiate a `ChatOpenAI` model

`ChatOpenAI` is LangChain's wrapper around the OpenAI chat completions endpoint. Key constructor parameters:

| Parameter | Default | Effect |
|---|---|---|
| `model` | `"gpt-3.5-turbo"` | Which OpenAI model to call |
| `temperature` | `0.7` | Randomness — 0 = deterministic, 2 = very random |
| `max_tokens` | None (model max) | Hard cap on output tokens |
| `timeout` | None | Seconds before the call raises a timeout error |
| `api_key` | `OPENAI_API_KEY` env var | Explicit key (avoid; prefer env var) |

LangChain implements a **chat model** interface (not a completion interface), so the primary method is `.invoke(messages)` — not `.complete(prompt)`.


In [ ]:
from langchain_openai import ChatOpenAI

# Instantiate the model — no network call happens here yet.
llm = ChatOpenAI(
    model="gpt-4o-mini",   # affordable, fast, good instruction-following
    temperature=0,          # deterministic output — good for demos and tests
    max_tokens=256,
)

print(f"Model: {llm.model_name}")
print(f"Temperature: {llm.temperature}")
print(f"Max tokens: {llm.max_tokens}")

### What just happened?
- **`ChatOpenAI` is lazy** — it stores configuration but makes no HTTP call at instantiation time.
- `model_name`, `temperature`, and `max_tokens` are stored as attributes you can inspect or override.
- LangChain validates your config at construction (e.g., invalid model names raise early).
- **`gpt-4o-mini`** is recommended for day-to-day LangChain experimentation: ~10× cheaper than `gpt-4o` with comparable instruction-following quality.


## Step 3 · LangChain message types

Every LangChain chat call is built from typed message objects that map directly to OpenAI's `role` field:

| LangChain class | OpenAI role | Purpose |
|---|---|---|
| `SystemMessage` | `system` | Persistent instructions — persona, constraints, output format |
| `HumanMessage` | `user` | What the end-user says |
| `AIMessage` | `assistant` | A previous model response (used for multi-turn history) |
| `FunctionMessage` | `function` | Result of a tool/function call (used in agent loops) |

**Official docs:** https://python.langchain.com/docs/concepts/messages/

The model receives a **list** of messages — the order matters. Convention: `[SystemMessage, HumanMessage, AIMessage, HumanMessage, …]`.


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Build a minimal conversation.
messages = [
    SystemMessage(content="You are a concise technical assistant. Answer in 1–2 sentences."),
    HumanMessage(content="What is LangChain's LCEL pipe operator used for?"),
]

# Inspect the message objects before sending.
for msg in messages:
    print(f"{type(msg).__name__:20s} | role={msg.type!r:12s} | {msg.content[:60]}")

### What just happened?
- Each message class wraps a `content` string and a `.type` property (`"system"`, `"human"`, `"ai"`).
- LangChain serialises these to the OpenAI `messages` array format automatically.
- **`SystemMessage` should always come first** in the list — the model processes it as global context.
- You can mix LangChain message objects with plain dicts (`{"role": "user", "content": "…"}`) in the same list.


## Step 4 · Make your first chat call and inspect the response

`llm.invoke(messages)` sends the messages to the API and returns an `AIMessage` object. The full response object contains more than just `content` — it includes token counts, model metadata, and finish reason.

Key attributes on the returned `AIMessage`:

| Attribute | Type | Contains |
|---|---|---|
| `.content` | `str` | The model's reply text |
| `.usage_metadata` | `dict` | `input_tokens`, `output_tokens`, `total_tokens` |
| `.response_metadata` | `dict` | `model_name`, `finish_reason`, `system_fingerprint` |
| `.id` | `str` | Unique completion ID (for logging / de-duplication) |


In [ ]:
import json

# --- MOCK BLOCK: remove this block and uncomment llm.invoke() when you have a real key ---
# This simulates the AIMessage structure so the rest of the notebook runs offline.
from langchain_core.messages import AIMessage

response = AIMessage(
    content="The LCEL pipe operator `|` chains Runnables together — output of the left side becomes input to the right side, building composable pipelines without boilerplate.",
    usage_metadata={"input_tokens": 42, "output_tokens": 28, "total_tokens": 70},
    response_metadata={"model_name": "gpt-4o-mini", "finish_reason": "stop", "system_fingerprint": "mock"},
    id="chatcmpl-mock-0001",
)
# --- end mock block ---

# Uncomment this to make a real API call:
# response = llm.invoke(messages)

print("=== content ===")
print(response.content)
print()
print("=== usage_metadata ===")
print(json.dumps(response.usage_metadata, indent=2))
print()
print("=== response_metadata ===")
print(json.dumps(response.response_metadata, indent=2))

### What just happened?
- `.content` is the plain-text answer — the only field most tutorials mention, but not the only useful one.
- **`usage_metadata`** lets you track token spend; `input_tokens + output_tokens = total_tokens`.
- **`response_metadata`** contains `finish_reason` — `"stop"` means natural end, `"length"` means `max_tokens` was hit (truncated output).
- The completion `id` is useful for logging to observability tools like LangSmith or Langfuse.


## Step 5 · Multi-turn conversation with `AIMessage` history

Chat models are **stateless** — each API call is independent. To maintain a conversation, you must **manually append previous messages** to the list. This is exactly what LangChain's memory classes automate, but understanding the raw mechanism first is essential.

Pattern:
```
[System, Human₁, AI₁, Human₂, AI₂, …, HumanN]
```
The model sees the entire history on each call.


In [ ]:
# Simulate a 2-turn conversation by appending the previous AI reply and a follow-up.
conversation = [
    SystemMessage(content="You are a concise technical assistant. Answer in 1–2 sentences."),
    HumanMessage(content="What is LangChain's LCEL pipe operator used for?"),
    AIMessage(content=response.content),   # the AI's first reply (from Step 4)
    HumanMessage(content="Give me a one-line code example."),
]

# Mock second response to avoid requiring a real key.
response2 = AIMessage(
    content='`chain = prompt | llm | StrOutputParser()` — this chains a prompt template, an LLM, and a parser into a single callable pipeline.',
    usage_metadata={"input_tokens": 90, "output_tokens": 32, "total_tokens": 122},
    response_metadata={"model_name": "gpt-4o-mini", "finish_reason": "stop"},
    id="chatcmpl-mock-0002",
)
# response2 = llm.invoke(conversation)  # uncomment with a real key

print("Turn 1 (tokens used):", response.usage_metadata.get("total_tokens"))
print("Turn 2 (tokens used):", response2.usage_metadata.get("total_tokens"))
print()
print("Assistant (turn 2):", response2.content)

### What just happened?
- **Token count grows** each turn because the full history is re-sent — important to track for cost control.
- `AIMessage(content=response.content)` converts our first response back into a message the model treats as its own prior output.
- This manual pattern is exactly what `ConversationBufferMemory` automates for you.
- For long conversations, use `ConversationSummaryMemory` to compress history instead of sending it raw.


## Step 6 · Streaming responses

For user-facing apps, streaming lets you show tokens as they arrive rather than waiting for the full response. LangChain exposes `.stream(messages)` which yields `AIMessageChunk` objects.

Streaming is important when:
- Response latency exceeds ~1 second (most GPT-4-class calls)
- You want to show a typing indicator in UI
- You need to cancel mid-generation (e.g., user clicks stop)


In [ ]:
# Demonstrate streaming with a mock generator (runs offline).
# With a real key: for chunk in llm.stream(messages): print(chunk.content, end="", flush=True)

from langchain_core.messages import AIMessageChunk

# Simulated stream: split a response into word-by-word chunks.
mock_reply = "LangChain connects your code to LLMs through a clean, composable interface."
mock_chunks = [AIMessageChunk(content=word + " ") for word in mock_reply.split()]

print("Streaming output (simulated):")
full_text = ""
for chunk in mock_chunks:
    print(chunk.content, end="", flush=True)  # print token as it arrives
    full_text += chunk.content

print("\n\nFull reassembled text:")
print(full_text.strip())

### What just happened?
- `.stream()` returns a generator — chunks arrive incrementally instead of all at once.
- Each `AIMessageChunk.content` is a partial string (often a single token or a few tokens).
- **Concatenating chunks** gives you the full response text, identical to what `.invoke()` would return.
- Streaming adds no extra cost — the same tokens are billed whether you stream or not.


In [ ]:
# Challenge: Build a simple cost estimator
# ─────────────────────────────────────────
# Given the token usage from a ChatOpenAI response, calculate the USD cost.
#
# gpt-4o-mini pricing (as of mid-2024):
#   Input:  $0.15 per 1M tokens  →  $0.00000015 per token
#   Output: $0.60 per 1M tokens  →  $0.00000060 per token
#
# TODO: write a function `estimate_cost(usage_metadata, model="gpt-4o-mini")` that returns
# a dict with keys: "input_cost", "output_cost", "total_cost" (all in USD, rounded to 8 dp).
# Then print a cost summary for `response` and `response2` from Steps 4 and 5.

PRICING = {
    "gpt-4o-mini": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
    "gpt-4o":      {"input": 5.00 / 1_000_000, "output": 15.00 / 1_000_000},
}

def estimate_cost(usage_metadata, model="gpt-4o-mini"):
    # Your solution here
    pass

# Test it:
# print(estimate_cost(response.usage_metadata))
# print(estimate_cost(response2.usage_metadata))

---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `ChatOpenAI` | LangChain's wrapper for OpenAI chat models — lazy init, use `.invoke(messages)` |
| `SystemMessage` | Sets the model's persona/constraints — always first in the messages list |
| `HumanMessage` | Represents user input |
| `AIMessage` | Represents a prior model reply — append to history for multi-turn conversations |
| `usage_metadata` | `input_tokens + output_tokens = total_tokens` — track this for cost control |
| `response_metadata.finish_reason` | `"stop"` = complete; `"length"` = truncated by `max_tokens` |
| `.stream()` | Yields `AIMessageChunk` objects token-by-token — same cost as `.invoke()` |
| API key safety | Load from `.env` via `python-dotenv`; never hardcode in notebooks |

> **Tip:** Set `OPENAI_API_KEY` in a `.env` file and load it with `python-dotenv` — never hardcode keys in notebooks.

---
## What's next
**Day 2** → Chat Models and Prompt Templates — learn to build reusable `ChatPromptTemplate` pipelines with variable substitution and few-shot examples.

Mark Day 1 complete in your [tracker](../index.html).
